## Homework: Agentic RAG
In this homework, we build a RAG system from scratch and then make it agentic, as we did in the module.

Instead of the course FAQ, our knowledge base is the course lessons themselves.

The course repository is organized by module. Each module is a top-level folder with a lessons/ subfolder of numbered markdown pages:

There are seven modules:

* ```01-agentic-rag```
* ```02-vector-search```
* ```03-orchestration```
* ```04-evaluation```
* ```05-monitoring```
* ```06-best-practices```
* ```07-project-example```

Each lesson page is a single markdown file. These pages are exactly what you read as you go through the course.

We'll fetch this data from GitHub and use it as the knowledge base for our RAG system.

(It's possible your answers won't match exactly. If so, select the closest one.)

### Setup
Prepare your environment the same way as in the module's Environment lesson.

This homework needs one extra library: ```gitsource```, which downloads files from a GitHub repository.

Install it:

```uv add gitsource```

For the LLM, we recommend OpenAI with ```gpt-5.4-mini```, but you can use any model and provider you like - just adapt the client and the usage fields accordingly.



### Preparation

First, we will pull the lesson pages straight from the course repository. We will use the commit ```8c1834d``` to make sure everyone works with the exact same data.

We will use ```gitsource``` for that:



In [53]:
def show(obj, max_items=5, max_chars_per_item=150):
    """Display object showing first N items, each truncated to max chars"""
    if isinstance(obj, list):
        for i, item in enumerate(obj[:max_items]):
            item_str = str(item)
            if len(item_str) > max_chars_per_item:
                print(f"{i+1}. {item_str[:max_chars_per_item]}...")
            else:
                print(f"{i+1}. {item_str}")
        if len(obj) > max_items:
            print(f"\n... and {len(obj) - max_items} more items")
    else:
        # For non-list objects, show first max_chars_per_item characters
        obj_str = str(obj)
        if len(obj_str) > max_chars_per_item:
            print(obj_str[:max_chars_per_item] + "...")
        else:
            print(obj_str)

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [54]:
show(files[:5])

1. RawRepositoryFile(filename='01-agentic-rag/lessons/01-intro.md', content='# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v...
2. RawRepositoryFile(filename='01-agentic-rag/lessons/02-environment.md', content='# Environment\n\nVideo: [Watch this lesson](https://www.youtube.com/wa...
3. RawRepositoryFile(filename='01-agentic-rag/lessons/03-rag.md', content='# RAG\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=JktYwBIDEr...
4. RawRepositoryFile(filename='01-agentic-rag/lessons/04-dataset.md', content='# The Course FAQ Dataset\n\nVideo: [Watch this lesson](https://www.youtube...
5. RawRepositoryFile(filename='01-agentic-rag/lessons/05-search.md', content='# Search\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=GYgp...


```GithubRepositoryDataReader``` downloads the entire repository and goes over all the files in it. Because we specify ```allowed_extensions={"md"}```, it only checks the markdown files.

We also pass a ```filename_filter``` so we don't grab every markdown file in the repo, like the top-level README. The lesson pages all live under a module's ```lessons/``` folder, so filtering on ```/lessons/``` keeps just those.

Each file has a ```parse()``` method that returns a dictionary with its ```filename``` and ```content```:

In [55]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [56]:
show(documents[:5])

1. {'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn t...
2. {'content': '# Environment\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=3U4gBrmkZyM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nFor t...
3. {'content': '# RAG\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=JktYwBIDErk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nWe run free Z...
4. {'content': '# The Course FAQ Dataset\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=Mx6EqvzVDz0&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XI...
5. {'content': '# Search\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=GYgpNKiuCJU&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\n## Search ...


### Q1. How many lesson pages?
How many lesson pages are in the dataset?  a) 24; b) 72; c) 240; d) 720



In [57]:
len(documents)

72

**There are 72 lesson pages in the dataset.**

### Q2. Indexing and searching
Index the documents with ```minsearch``` - make ```content``` a text field and ```filename``` a keyword field. Then search with this query:

"How does the agentic loop keep calling the model until it stops?"

What's the ```filename``` of the first result?

In [58]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [59]:
show(documents[:5])

1. {'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn t...
2. {'content': '# Environment\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=3U4gBrmkZyM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nFor t...
3. {'content': '# RAG\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=JktYwBIDErk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nWe run free Z...
4. {'content': '# The Course FAQ Dataset\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=Mx6EqvzVDz0&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XI...
5. {'content': '# Search\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=GYgpNKiuCJU&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\n## Search ...


In [60]:
len(documents)

72

In [61]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

show(search_results)

1. {'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\n...
2. {'content': '# ToyAIKit\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=PQpQOR3Un3w&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nThe hand...
3. {'content': '# Function Calling\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=CeEki_0mdGo&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\n...
4. {'content': "# Agents\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=6uG4_Ivv60E&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn Part 1 ...
5. {'content': "# Other Frameworks\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=4yiCbKX9RhI&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\n...


**The first result is ```01-agentic-rag/lessons/14-agentic-loop.md```**

### Q3. RAG
Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:



In [62]:
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

```RAGBase``` was written for the FAQ schema (```section```/```question```/```answer```), while our documents have ```filename``` and ```content```.

Two solutions are possible:

* Implement the RAG flow yourself
* Take ```RAGBase``` and change the parts related to the FAQ schema - ```search``` (to use our index) and ```build_context```

Build a RAG over the index from Q2 and answer the query:

"How does the agentic loop keep calling the model until it stops?"

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request? The options are:  
* 700
* 7000
* 70000
* 700000

We count input tokens instead of price because the cost depends on the model and provider you use, but the size of the prompt we send is the same for everyone.

Most LLM APIs report token usage on the response object (e.g. response.usage.input_tokens / prompt_tokens). We'll read the input tokens from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, llm returns only the text. Modify it to return the whole response, and change rag to return both the answer and usage (as a tuple or create a small dataclass for that).

Note: for this question and the next ones, if your answer doesn't match exactly, just select the closest option - especially if you use a different model or a different LLM provider.

In [63]:
from dotenv import load_dotenv
load_dotenv()

from rag_helper1 import RAGBase
from openai import OpenAI

openai_client = OpenAI()

# Customize RAGBase for this schema
class CustomRAGBase(RAGBase):
    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)
    
    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"File: {doc['filename']}")
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()
    
    def rag_with_response(self, query):
        """Returns both answer and response object"""
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]
        
        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=input_messages
        )
        
        answer = response.choices[0].message.content
        return answer, response

assistant = CustomRAGBase(index=index, llm_client=openai_client)
answer, response = assistant.rag_with_response("How does the agentic loop keep calling the model until it stops?")
print(answer)

The loop keeps calling the model by using a `while True` loop and a flag like `has_function_calls`.

Each iteration:
1. Send the full `messages` history to the model.
2. If the model returns a `function_call`, run the tool and append the result to `messages`.
3. If there are no function calls, break out of the loop.

So the stop condition is simply: **if the model returns no function calls, the loop ends**.


In [64]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.prompt_tokens * input_price +
    response.usage.completion_tokens * output_price
)

print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Cost: ${cost}")

Input tokens: 7126
Cost: $0.0057945


**According to this run, we sent 7126 input tokens.  The closest option in the multiple-choice answers is 7000, so that's the answer I'm selecting.**

### Q4. Chunking
The lesson pages are long - some are thousands of characters. Long documents make retrieval less precise: a match deep inside a page still pulls in the whole page. A common fix is chunking: split each page into smaller, overlapping pieces and index those instead.

```gitsource``` has a helper for this: ```chunk_documents```. It uses a sliding window - a window of ```size``` characters slides across the text in steps of ```step``` characters, and each window position becomes one chunk:

In [65]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

With ```size=2000``` and ```step=1000``` (you can see the implementation here):

* Each chunk is a window of ```size``` characters of the page.
* The window moves forward by ```step``` characters between chunks. Since ```step``` is smaller than ```size```, consecutive chunks overlap by ```size - step``` (1000) characters, so a passage split across a boundary still appears whole in one of the chunks.
* Every chunk keeps the original fields (```filename```) and adds ```start``` (the offset in the page) and ```content``` (the chunk text).
How many chunks do you get?

* 70
* 295
* 1100
* 4500

In [66]:
len(chunks)

295

**We get 295 chunks from this process.**

### Q5. RAG with chunking
Chunking makes each request smaller, because we send a smaller context to the LLM. Let's measure that.

Index the chunks from Q4 (same as before: ```content``` as a text field, ```filename``` as a keyword field), point your RAG at the chunk index, and answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

* about the same
* 3× fewer
* 10× fewer
* 30× fewer


In [67]:
index.fit(chunks)

In [68]:
show(chunks[:5])

1. {'start': 0, 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0...
2. {'start': 1000, 'content': 'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\...
3. {'start': 2000, 'content': 'wrong.\n\n## The project\n\nRAG solves these problems by giving the LLM relevant documents at\nquestion time. We don\'t ho...
4. {'start': 0, 'content': '# Environment\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=3U4gBrmkZyM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0X...
5. {'start': 1000, 'content': "nitialize it:\n\n```bash\nmkdir llm-zoomcamp-code\ncd llm-zoomcamp-code\nuv init\n```\n\nThis creates a `pyproject.toml` a...


In [69]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

show(search_results)

1. {'start': 4000, 'content': 'while` loop. The loop keeps calling the model until\nit returns a response without any function calls. We also keep an\nit...
2. {'start': 5000, 'content': 'ees the result on the next turn.\nThe loop stops when the model returns a final answer with no more tool\ncalls.\n\nWe don...
3. {'start': 0, 'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK...
4. {'start': 4000, 'content': 'esult.\n\nThe `result` is a `LoopResult` with `all_messages` (the full\nconversation), token counts, and `cost` (computed ...
5. {'start': 3000, 'content': '   developer_prompt=instructions,\n    chat_interface=chat_interface,\n    llm_client=OpenAIClient(model="gpt-5.4-mini")\n...


In [70]:
assistant = CustomRAGBase(index=index, llm_client=openai_client)
answer, response = assistant.rag_with_response("How does the agentic loop keep calling the model until it stops?")
print(answer)

The loop keeps calling the model inside a `while True` loop, and after each turn it checks whether any `function_call` items appeared in the model’s output.

- If the model returns one or more function calls, the code runs them, appends the tool results to `messages`, and loops again.
- If the model returns a normal `message` with no function calls, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is: **no function calls in the current response**.


In [71]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.prompt_tokens * input_price +
    response.usage.completion_tokens * output_price
)

print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Cost: ${cost}")

Input tokens: 2309
Cost: $0.00223125


**The number of input tokens used is 2309 -- about 3x fewer than in the non-chunked example.**

### Q6. Turning it into an agent
So far search runs once, with the exact query. Let's make it agentic: give the LLM a ```search``` tool and let it decide when (and what) to search. We suggest toyaikit, the small agent library from the module, but you can use anything you like - the OpenAI Agents SDK, PydanticAI, LangChain, or a hand-written loop.  

Create a search function that uses the chunk index. Give it a type hint and a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your search tool and run it (with toyaikit, the same way as in the ToyAIKit lesson). Use these instructions for the agent (they nudge it to search a few times):

"You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

Ask it:

"How does the agentic loop work, and how is it different from plain RAG?"

The agent decides on its own when to search and when to answer. Count how many times it called the search tool.

How many times did the agent call search?

Note: the agent decides this itself, so it varies a little between runs - pick the closest option. We measured this with OpenAI gpt-5.4-mini; with a different model or provider the number may differ, so keep that in mind.

* 0
* 4
* 10
* 20


In [72]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [73]:
def search(query: str) -> dict[str, str]:
    """
    Search the database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        # boost_dict={"question": 3.0, "section": 0.5},
        # filter_dict={"course": "llm-zoomcamp"}
    )

In [74]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [75]:
show(agent_tools.get_tools())

1. {'type': 'function', 'name': 'search', 'description': 'Search the database for entries matching the given query.', 'parameters': {'type': 'object', 'p...


In [76]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics; offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using the course lessons pages indexed, don't do it yourself. Only use the 
facts from the course lessons pages.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [77]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [81]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback
)

# Extract just the final answer text (no wrapper)
if result.all_messages:
    for msg in reversed(result.all_messages):
        if hasattr(msg, 'text'):
            print(msg.text)
            break

-> Response received


-> Response received


-> Response received


**This time, the agent called search four times.** I've run this cell several times and have found that the agent has called search between 2 and 4 times.

### Learning in Public
We encourage everyone to share what they learned. This is called "learning in public".

#### Why learn in public?
* Accountability: Sharing your progress creates commitment and motivation to continue
* Feedback: The community can provide valuable suggestions and corrections
* Networking: You'll connect with like-minded people and potential collaborators
* Documentation: Your posts become a learning journal you can reference later
* Opportunities: Employers and clients often discover talent through public learning
* You can read more about the benefits here and in the course's learning in public guide.

Don't worry about being perfect. Everyone starts somewhere, and people love following genuine learning journeys!